# Duplex KV Cache Walkthrough

This notebook is meant for learning, not just testing. It starts from the most basic question:

> How does a transformer decide which previous words matter, and why can we cache part of that work?

Then it connects that idea to the duplex system prompt update in this repo.

The path is:

1. Attention from zero: queries, keys, values, and weighted sums.
2. Why generation repeats work without a cache.
3. What a KV cache stores.
4. How the duplex cache is laid out.
5. How `update_system_prompt()` rebuilds only the protected prompt span.
6. What is approximate about preserving old conversation units.

Mental model for the duplex cache layout:

```text
[system prefix + optional ref audio] [previous summary] [system suffix] [live conversation units]
```

When the system prompt changes, the implementation rebuilds the protected system span and keeps later unit cache entries. If the new system span has a different length, preserved unit keys are RoPE-reindexed so their absolute positions move to the new location.

## 1. Colab Setup

If you opened this notebook from GitHub, Colab opened the notebook file but did not automatically clone the repository into `/content`. The setup cell below clones the repo if needed and then adds `src/` to `sys.path`.

If you use a fork or a different checkout path, change `REPO_PATH` and `REPO_URL`.

In [ ]:
%pip -q install transformers pytest matplotlib


In [ ]:
from pathlib import Path
import sys

REPO_PATH = Path('/content/minicpm-o-agent')
REPO_URL = 'https://github.com/erkamkavak/minicpm-o-agent.git'

if not REPO_PATH.exists():
    !git clone {REPO_URL} {REPO_PATH}

sys.path.insert(0, str(REPO_PATH / 'src'))
print('Repo path:', REPO_PATH)
print('Source path added:', REPO_PATH / 'src')


In [ ]:
import torch
from types import SimpleNamespace
from IPython.display import HTML, display
import matplotlib.pyplot as plt

torch.set_printoptions(precision=2, sci_mode=False)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())


## 2. Attention From Zero

A transformer does not read a prompt as one single blob. It reads a sequence of token vectors.

For each token, the model creates three projections:

```text
Q = query  = what this token is looking for
K = key    = what this token offers as a match label
V = value  = the information/content this token contributes
```

An attention layer answers this question for the current token:

> Looking at the keys of previous tokens, which values should I mix together?

A simple analogy:

- Query: a search question.
- Keys: index cards describing what each earlier token contains.
- Values: the actual notes behind those index cards.
- Attention weights: how much each note should matter.

In [ ]:
def show_attention_concept_diagram():
    svg = r"""
    <svg width="920" height="330" viewBox="0 0 920 330" xmlns="http://www.w3.org/2000/svg">
      <defs>
        <marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto" markerUnits="strokeWidth">
          <path d="M0,0 L0,6 L9,3 z" fill="#444" />
        </marker>
      </defs>
      <style>
        .box { fill: #f7f7f7; stroke: #333; stroke-width: 1.2; rx: 8; }
        .q { fill: #fff2cc; }
        .k { fill: #d9ead3; }
        .v { fill: #d9eaf7; }
        .out { fill: #eadcf8; }
        .txt { font-family: sans-serif; font-size: 15px; fill: #222; }
        .small { font-size: 12px; fill: #444; }
        .title { font-size: 18px; font-weight: 700; }
        .arrow { stroke: #444; stroke-width: 1.4; fill: none; marker-end: url(#arrow); }
      </style>
      <text x="30" y="32" class="txt title">One attention step for the current token</text>

      <rect x="35" y="65" width="120" height="56" class="box"/>
      <text x="63" y="90" class="txt">current token</text>
      <text x="64" y="110" class="txt small">the token we are computing</text>

      <rect x="220" y="55" width="110" height="48" class="box q"/>
      <text x="255" y="84" class="txt">Q</text>
      <text x="236" y="120" class="txt small">what am I looking for?</text>

      <rect x="410" y="55" width="110" height="48" class="box k"/>
      <text x="445" y="84" class="txt">K</text>
      <text x="385" y="120" class="txt small">match labels from previous tokens</text>

      <rect x="410" y="175" width="110" height="48" class="box v"/>
      <text x="445" y="204" class="txt">V</text>
      <text x="392" y="240" class="txt small">content from previous tokens</text>

      <rect x="620" y="87" width="150" height="72" class="box"/>
      <text x="650" y="118" class="txt">softmax(Q · K)</text>
      <text x="656" y="140" class="txt small">attention weights</text>

      <rect x="805" y="143" width="90" height="56" class="box out"/>
      <text x="829" y="172" class="txt">output</text>
      <text x="819" y="192" class="txt small">weighted V</text>

      <path d="M155 93 L220 80" class="arrow"/>
      <path d="M330 80 L410 80" class="arrow"/>
      <path d="M520 80 L620 110" class="arrow"/>
      <path d="M520 200 L620 140" class="arrow"/>
      <path d="M770 125 L805 165" class="arrow"/>

      <text x="35" y="288" class="txt small">Important for caching: during generation, old K and V do not change. We can keep them and only compute Q/K/V for the newest token.</text>
    </svg>
    """
    display(HTML(svg))

show_attention_concept_diagram()

### A Tiny Numeric Attention Example

The next cell does a real attention calculation with tiny hand-made vectors.

We pretend the current token is asking a question with query vector `q`. Each previous token has a key vector `K` and value vector `V`.

The formula is:

```text
scores  = q · K
weights = softmax(scores)
output  = weights · V
```

Do not worry about the exact numbers yet. Watch the shape of the process:

1. Similar query/key pairs get larger scores.
2. `softmax` turns scores into percentages that sum to 1.
3. The output is a weighted mixture of value vectors.

In [ ]:
previous_tokens = ["system", "user", "audio", "assistant"]

# Query for the current token: what it is looking for.
q = torch.tensor([1.0, 0.2])

# Keys: match labels for previous tokens.
K = torch.tensor([
    [0.2, 1.0],  # system
    [1.0, 0.1],  # user
    [0.8, 0.3],  # audio
    [0.1, 0.9],  # assistant
])

# Values: content vectors that will be mixed if their key matches.
V = torch.tensor([
    [10.0, 0.0],
    [0.0, 10.0],
    [5.0, 5.0],
    [8.0, 2.0],
])

scores = K @ q
weights = torch.softmax(scores, dim=0)
output = weights @ V

print('query q:', q.tolist())
print('\nScores = K @ q')
for token, score in zip(previous_tokens, scores.tolist()):
    print(f'{token:>9}: {score:.3f}')

print('\nAttention weights = softmax(scores)')
for token, weight in zip(previous_tokens, weights.tolist()):
    print(f'{token:>9}: {weight:.3f}')

print('\nOutput = weighted sum of values:', output.tolist())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].bar(previous_tokens, scores.tolist(), color='#9fc5e8')
axes[0].set_title('Raw match scores')
axes[0].set_ylabel('q dot key')
axes[0].tick_params(axis='x', rotation=25)

axes[1].bar(previous_tokens, weights.tolist(), color='#b6d7a8')
axes[1].set_title('Softmax attention weights')
axes[1].set_ylabel('share of attention')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

## 3. Why KV Cache Exists During Generation

When generating text/audio tokens, the model produces one new token at a time.

Without a KV cache, every new token would force the model to recompute keys and values for the entire prefix again:

```text
step 1: process [prompt]
step 2: process [prompt + token1]
step 3: process [prompt + token1 + token2]
step 4: process [prompt + token1 + token2 + token3]
```

But old keys and values are stable. The token `system` has the same K/V vectors at generation step 1 and step 100. So the model can cache old K/V and only compute the new token's Q/K/V.

```text
prefill once: cache K/V for [prompt]
step 1: compute new token, append its K/V to cache
step 2: compute next token, append its K/V to cache
step 3: compute next token, append its K/V to cache
```

In [ ]:
def show_cache_saves_work_diagram(prompt_len=5, generated_steps=5):
    no_cache = [prompt_len + i for i in range(generated_steps)]
    with_cache = [1 for _ in range(generated_steps)]
    with_cache[0] = prompt_len

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
    steps = [f'step {i+1}' for i in range(generated_steps)]

    axes[0].bar(steps, no_cache, color='#f4cccc')
    axes[0].set_title('Without KV cache')
    axes[0].set_ylabel('tokens processed by LLM')
    axes[0].tick_params(axis='x', rotation=25)
    for i, v in enumerate(no_cache):
        axes[0].text(i, v + 0.1, str(v), ha='center')

    axes[1].bar(steps, with_cache, color='#d9ead3')
    axes[1].set_title('With KV cache')
    axes[1].tick_params(axis='x', rotation=25)
    for i, v in enumerate(with_cache):
        axes[1].text(i, v + 0.1, str(v), ha='center')

    plt.suptitle('Caching avoids recomputing old K/V every generation step')
    plt.tight_layout()
    plt.show()

    print('Total token-work without cache:', sum(no_cache))
    print('Total token-work with cache:   ', sum(with_cache))

show_cache_saves_work_diagram()

### What Is Actually Stored?

A KV cache does not store text. It stores tensors.

For each transformer layer, it stores:

```text
key_cache[layer]:   [batch, heads, cached_tokens, head_dim]
value_cache[layer]: [batch, heads, cached_tokens, head_dim]
```

The `cached_tokens` axis is the important one in this notebook. That axis grows as the conversation grows.

A simplified cache after prompt prefill and two generated units might look like this:

```text
position:  0      1      2      3      4      5      6
content:  sys0   sys1   sys2   user0  user1  asst0  asst1
cache:    K/V    K/V    K/V    K/V    K/V    K/V    K/V
```

When we say “slice the cache,” we mean slicing tensors along that cached-token axis.

In [ ]:
def show_cache_layout_diagram():
    svg = r"""
    <svg width="930" height="260" viewBox="0 0 930 260" xmlns="http://www.w3.org/2000/svg">
      <style>
        .txt { font-family: sans-serif; font-size: 14px; fill: #222; }
        .small { font-size: 12px; fill: #444; }
        .sys { fill: #fff2cc; stroke: #555; }
        .prev { fill: #eadcf8; stroke: #555; }
        .suf { fill: #fce5cd; stroke: #555; }
        .unit { fill: #d9ead3; stroke: #555; }
        .label { font-weight: 700; }
      </style>
      <text x="30" y="28" class="txt label">Duplex KV cache layout</text>
      <text x="30" y="52" class="small">Each rectangle is one cached token position. In the real model, every position contains key/value tensors for every transformer layer.</text>

      <g transform="translate(30,85)">
        <rect x="0" y="0" width="90" height="58" class="sys"/><text x="20" y="33" class="txt">prefix</text><text x="28" y="78" class="small">0..2</text>
        <rect x="90" y="0" width="90" height="58" class="sys"/><text x="110" y="33" class="txt">prefix</text>
        <rect x="180" y="0" width="90" height="58" class="sys"/><text x="200" y="33" class="txt">prefix</text>
        <rect x="270" y="0" width="120" height="58" class="prev"/><text x="292" y="33" class="txt">previous</text><text x="305" y="78" class="small">optional</text>
        <rect x="390" y="0" width="90" height="58" class="suf"/><text x="417" y="33" class="txt">suffix</text>
        <rect x="480" y="0" width="90" height="58" class="unit"/><text x="505" y="33" class="txt">unit</text><text x="497" y="78" class="small">live history</text>
        <rect x="570" y="0" width="90" height="58" class="unit"/><text x="595" y="33" class="txt">unit</text>
        <rect x="660" y="0" width="90" height="58" class="unit"/><text x="685" y="33" class="txt">unit</text>
      </g>

      <text x="30" y="205" class="txt">Protected system span = prefix + previous + suffix</text>
      <text x="30" y="228" class="txt">Preserved unit span = later conversation/audio/video units that remain after the system prompt update</text>
    </svg>
    """
    display(HTML(svg))

show_cache_layout_diagram()

## 4. Build A Visible Fake KV Cache

Now that attention has a shape, we can make a fake cache that is easy to inspect.

Real K/V tensors contain many floating point numbers that are hard to read. This toy cache uses readable numbers so you can see where each token goes.

A common shape is:

```text
key/value: [batch, heads, sequence_length, head_dim]
```

In the toy helper below:

- keys encode `token_id + position / 100`
- values encode `token_id - position / 100`

That is not real model math. It is just a microscope for understanding layout, slicing, and concatenation.

In [ ]:
def make_visible_cache(token_ids, start_position=0, heads=1, head_dim=4):
    """Create a readable fake KV cache for one layer.

    Keys encode token_id + absolute position.
    Values encode token_id - absolute position.
    Real models use learned projections, but the cache shape and indexing idea is the same.
    """
    token_ids = torch.tensor(token_ids, dtype=torch.float32)
    positions = torch.arange(start_position, start_position + len(token_ids), dtype=torch.float32)
    base = token_ids[:, None].repeat(1, head_dim)
    pos = positions[:, None].repeat(1, head_dim)
    keys = (base + pos / 100).reshape(1, heads, len(token_ids), head_dim)
    values = (base - pos / 100).reshape(1, heads, len(token_ids), head_dim)
    return ((keys, values),)

def cache_len(cache):
    if cache is None:
        return 0
    return cache[0][0].shape[2]

def slice_cache(cache, start, end=None):
    return tuple((k[:, :, start:end, :].clone(), v[:, :, start:end, :].clone()) for k, v in cache)

def concat_caches(*parts):
    parts = [p for p in parts if p is not None and cache_len(p) > 0]
    if not parts:
        return None
    layers = []
    for layer_index in range(len(parts[0])):
        keys = torch.cat([p[layer_index][0] for p in parts], dim=2)
        values = torch.cat([p[layer_index][1] for p in parts], dim=2)
        layers.append((keys, values))
    return tuple(layers)

def show_cache(cache, labels=None, title='cache'):
    keys, values = cache[0]
    print(f'\n{title}: length={cache_len(cache)}, shape={tuple(keys.shape)}')
    for i in range(cache_len(cache)):
        label = labels[i] if labels and i < len(labels) else str(i)
        k = keys[0, 0, i, :].tolist()
        v = values[0, 0, i, :].tolist()
        print(f'{i:02d} {label:<12} key={k} value={v}')

system_tokens = [10, 11, 12]
unit_tokens = [70, 71]
cache = concat_caches(
    make_visible_cache(system_tokens, start_position=0),
    make_visible_cache(unit_tokens, start_position=len(system_tokens)),
)
show_cache(cache, ['sys0', 'sys1', 'sys2', 'unit0', 'unit1'], 'initial cache')

## 5. The System Prompt Update Operation

The duplex update does cache surgery.

Starting point:

```text
[old protected system span] [live units]
```

Target:

```text
[new protected system span] [same live units]
```

The high-level steps are:

1. Compute where the old protected system span ends.
2. Slice out the later unit cache.
3. Build embeddings for the new system prompt, optional reference audio, previous summary, and suffix.
4. Run the LLM once on that new protected span to get a fresh system KV cache.
5. If the new protected span length changed, reindex preserved unit keys for their new positions.
6. Concatenate fresh system cache + preserved unit cache.

In [ ]:
old_system_end = 3
units_cache = slice_cache(cache, old_system_end, cache_len(cache))
show_cache(units_cache, ['unit0', 'unit1'], 'sliced units before update')

new_system_tokens = [20, 21, 22, 23, 24]
new_system_cache = make_visible_cache(new_system_tokens, start_position=0)

# This concat demonstrates the layout change. Real code also reindexes RoPE keys when positions move.
updated_cache_without_rope = concat_caches(new_system_cache, units_cache)
show_cache(
    updated_cache_without_rope,
    ['new_sys0', 'new_sys1', 'new_sys2', 'new_sys3', 'new_sys4', 'unit0', 'unit1'],
    'updated layout before RoPE reindex idea',
)

## 6. Why Position Reindexing Exists

MiniCPM-style LLMs use rotary positional embeddings, usually called RoPE.

The short version:

- Values mostly carry content.
- Keys carry content plus position information.
- If a cached token moves from position `3` to position `5`, its key needs to be adjusted for the new position.

In our update, this can happen because the new system prompt may be longer or shorter than the old one.

Example:

```text
old: [system length 3] [unit0 at pos 3] [unit1 at pos 4]
new: [system length 5] [unit0 at pos 5] [unit1 at pos 6]
```

The repo does this in `StreamDecoder._reindex_rope_for_cache()`.

In [ ]:
def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)

def apply_tiny_rope(x, positions, theta=10000.0):
    """Small RoPE helper for demonstration, not a drop-in replacement for every model."""
    dim = x.shape[-1]
    inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2, dtype=x.dtype) / dim))
    freqs = torch.outer(positions.to(x.dtype), inv_freq)
    emb = torch.cat([freqs, freqs], dim=-1)
    cos = emb.cos().view(1, 1, len(positions), dim)
    sin = emb.sin().view(1, 1, len(positions), dim)
    return (x * cos) + (rotate_half(x) * sin)

def rope_reindex_keys(keys, old_start, new_start, length):
    old_positions = torch.arange(old_start, old_start + length)
    new_positions = torch.arange(new_start, new_start + length)

    # Approximate the idea used by the real method:
    # remove old rotation, then apply new rotation.
    unrotated = apply_tiny_rope(keys, old_positions, theta=10000.0)
    reindexed = apply_tiny_rope(unrotated, new_positions, theta=10000.0)
    return reindexed

old_unit_keys = units_cache[0][0]
moved_unit_keys = rope_reindex_keys(old_unit_keys, old_start=3, new_start=5, length=2)

print('old unit key at old position 3:', old_unit_keys[0, 0, 0].tolist())
print('same unit key after position move 3 -> 5:', moved_unit_keys[0, 0, 0].tolist())
print('\nNotice: values are unchanged by RoPE reindexing; only keys carry this positional rotation.')

## 7. Run The Repo's Actual Update Method With A Fake LLM

Now we leave the pure toy helpers and call the repo's real `StreamDecoder.update_system_prompt()` method.

We still use a fake LLM so this runs without real MiniCPM weights. The fake LLM is deliberately simple:

- `key = embedding + position`
- `value = embedding - position`

That makes it easy to inspect which parts were rebuilt and which parts were preserved.

The important thing here is not the fake math. The important thing is that this calls the actual update method from `src/minicpmo_demo/model/runtime/stream_decoder.py`.

In [ ]:
if not REPO_PATH.exists():
    raise RuntimeError('Set REPO_PATH to your cloned/uploaded repo before running this cell.')

from minicpmo_demo.model.runtime.stream_decoder import StreamDecoder

class FakeTokenizer:
    eos_token_id = 0
    unk_token_id = -1
    all_special_ids = []
    all_special_tokens = []

    def convert_tokens_to_ids(self, token):
        return {'<|chunk_eos|>': 1, '<|chunk_tts_eos|>': 2, '<|turn_eos|>': 3, '<|speak|>': 4}.get(token, 99)

    def encode(self, text, add_special_tokens=False):
        return [ord(c) % 97 for c in text]

    def decode(self, token_ids, skip_special_tokens=False):
        return ''.join(str(i) for i in token_ids)

class FakeEmbeddingModel:
    def __init__(self, hidden_size):
        self.embedding = torch.nn.Embedding(256, hidden_size)
        with torch.no_grad():
            weight = torch.arange(256 * hidden_size, dtype=torch.float32).reshape(256, hidden_size)
            self.embedding.weight.copy_(weight / 1000.0)

    def embed_tokens(self, token_ids):
        return self.embedding(token_ids)

class FakeLLM:
    device = torch.device('cpu')

    def __init__(self, hidden_size=4):
        self.config = SimpleNamespace(hidden_size=hidden_size, rope_theta=10000.0)
        self.model = FakeEmbeddingModel(hidden_size)

    def __call__(self, inputs_embeds, position_ids, past_key_values=None, use_cache=True, return_dict=True):
        del use_cache, return_dict
        positions = position_ids.to(inputs_embeds.dtype).unsqueeze(-1)
        keys = inputs_embeds.unsqueeze(1) + positions
        values = inputs_embeds.unsqueeze(1) - positions
        current = ((keys, values),)
        if past_key_values is None:
            cache = current
        else:
            cache = ((torch.cat([past_key_values[0][0], current[0][0]], dim=2), torch.cat([past_key_values[0][1], current[0][1]], dim=2)),)
        return SimpleNamespace(past_key_values=cache)

def make_repo_style_cache(length, hidden_size=4):
    values = torch.arange(length * hidden_size, dtype=torch.float32).reshape(1, 1, length, hidden_size)
    keys = values + 0.5
    return ((keys, values),)

decoder = StreamDecoder(FakeLLM(), FakeTokenizer())

# Initial layout: [prefix length 2] [previous length 1] [suffix length 1] [unit length 2]
decoder.cache = make_repo_style_cache(6)
decoder._preserve_prefix_length = 2
decoder._previous_token_ids = [30]
decoder._previous_content_length = 1
decoder._suffix_token_ids = [40]
decoder._system_preserve_length = 4
decoder._unit_history = [{'unit_id': 0, 'length': 2, 'type': 'audio'}]

old_unit_values = decoder.cache[0][1][:, :, 4:6, :].clone()

show_cache(decoder.cache, ['prefix0', 'prefix1', 'previous', 'suffix', 'unit0', 'unit1'], 'repo decoder before update')

In [ ]:
ref_audio_embeds = torch.ones(2, 4)

updated = decoder.update_system_prompt(
    new_prefix_token_ids=[10, 11, 12],
    new_suffix_token_ids=[50],
    new_ref_audio_embeds=ref_audio_embeds,
)

print('updated:', updated)
print('new cache length:', decoder.get_cache_length())
print('new protected prefix length:', decoder._preserve_prefix_length)
print('previous length:', decoder._previous_content_length)
print('suffix ids:', decoder._suffix_token_ids)
print('system preserve length:', decoder._system_preserve_length)

show_cache(
    decoder.cache,
    ['new_prefix0', 'new_prefix1', 'new_prefix2', 'ref0', 'ref1', 'previous', 'new_suffix', 'unit0', 'unit1'],
    'repo decoder after update',
)

print('\nPreserved unit values moved from old positions 4:6 to new positions 7:9:')
print(torch.equal(decoder.cache[0][1][:, :, 7:9, :], old_unit_values))
print('old unit values:')
print(old_unit_values[0, 0])
print('new unit values at the end:')
print(decoder.cache[0][1][0, 0, 7:9])

## 8. Compare Cache Surgery vs Full Replay

There are two different questions here:

1. **Speed / compute:** how much work do we save by rebuilding only the protected system span and preserving unit cache?
2. **Divergence / accuracy:** how different is cache surgery from a mathematically exact full replay under the new prompt?

These are separate. Cache surgery can be much faster while still not being exactly equivalent to full replay.

### Compute Intuition

A full replay after changing the prompt means:

```text
run LLM again on: [new system] [previous summary] [suffix] [all old units]
```

Cache surgery means:

```text
run LLM again on: [new system] [previous summary] [suffix]
then linearly adjust/preserve: [all old units]
```

For attention, full replay gets expensive because each token can attend to many earlier tokens. A rough prefill attention-pair count is:

```text
full replay:     total_tokens * (total_tokens + 1) / 2
system rebuild:  new_system_tokens * (new_system_tokens + 1) / 2
unit reindex:    linear in number of preserved unit cache entries
```

This is only an estimate, not an exact GPU profiler. But it helps show why preserving thousands of old unit tokens matters.

In [ ]:
def estimate_attention_pairs(system_len, units_len):
    total = system_len + units_len
    full_replay_pairs = total * (total + 1) // 2
    system_rebuild_pairs = system_len * (system_len + 1) // 2
    return full_replay_pairs, system_rebuild_pairs

system_len = 256
unit_lengths = [0, 256, 512, 1024, 2048, 4096, 8192]

print(f'Assume new protected system span length = {system_len} tokens')
print('\nunits_len | full_replay_pairs | surgery_system_pairs | rough_pair_saving')
print('-' * 72)
for units_len in unit_lengths:
    full_pairs, surgery_pairs = estimate_attention_pairs(system_len, units_len)
    saving = full_pairs / max(surgery_pairs, 1)
    print(f'{units_len:9d} | {full_pairs:17,d} | {surgery_pairs:20,d} | {saving:15.1f}x')

fig, ax = plt.subplots(figsize=(8, 4))
full_values = []
surgery_values = []
for units_len in unit_lengths:
    full_pairs, surgery_pairs = estimate_attention_pairs(system_len, units_len)
    full_values.append(full_pairs)
    surgery_values.append(surgery_pairs)

ax.plot(unit_lengths, full_values, marker='o', label='full replay attention pairs')
ax.plot(unit_lengths, surgery_values, marker='o', label='system rebuild pairs')
ax.set_xlabel('preserved unit tokens')
ax.set_ylabel('rough attention pair count')
ax.set_title('Why cache surgery becomes attractive as history grows')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

### A Runnable Toy Timing Benchmark

The next cell is not MiniCPM. It is a small synthetic benchmark that imitates the shape of the work:

- Full replay does a square attention-like matrix multiply over all tokens.
- Cache surgery does the square work only over the new system span, then a linear operation over preserved unit cache.

This is intentionally simple so it can run on CPU or GPU in Colab. The absolute milliseconds are not the point. The trend is the point.

In [ ]:
import time

def time_ms(fn, repeats=5):
    # Warmup
    fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000)
    return min(times)

benchmark_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('benchmark_device:', benchmark_device)

hidden = 128
system_len = 256
unit_lengths = [256, 512, 1024, 2048]
if benchmark_device.type == 'cuda':
    unit_lengths.append(4096)

results = []
for units_len in unit_lengths:
    total_len = system_len + units_len
    full_x = torch.randn(total_len, hidden, device=benchmark_device)
    system_x = torch.randn(system_len, hidden, device=benchmark_device)
    units_cache = torch.randn(units_len, hidden, device=benchmark_device)

    def full_replay_work():
        scores = full_x @ full_x.T
        probs = torch.softmax(scores / hidden**0.5, dim=-1)
        out = probs @ full_x
        return out

    def surgery_work():
        scores = system_x @ system_x.T
        probs = torch.softmax(scores / hidden**0.5, dim=-1)
        system_out = probs @ system_x
        # Stand-in for linear RoPE reindex/preserve work on unit keys.
        moved_units = units_cache * 1.0001
        return system_out, moved_units

    full_ms = time_ms(full_replay_work, repeats=3)
    surgery_ms = time_ms(surgery_work, repeats=3)
    results.append((units_len, full_ms, surgery_ms, full_ms / surgery_ms))

print('\nunits_len | full_replay_ms | surgery_ms | speedup')
print('-' * 58)
for units_len, full_ms, surgery_ms, speedup in results:
    print(f'{units_len:9d} | {full_ms:14.2f} | {surgery_ms:10.2f} | {speedup:7.2f}x')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([r[0] for r in results], [r[1] for r in results], marker='o', label='full replay toy work')
ax.plot([r[0] for r in results], [r[2] for r in results], marker='o', label='cache surgery toy work')
ax.set_xlabel('preserved unit tokens')
ax.set_ylabel('milliseconds, lower is better')
ax.set_title('Toy timing comparison')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 9. Compare Divergence From Full Replay

Now the subtle part: cache surgery is not exactly the same as full replay.

Why?

When old units were created, their K/V tensors were computed in the context of the old system prompt. If we preserve those unit K/V tensors after changing the system prompt, they still carry some old-context influence.

A full replay would recompute those units under the new system prompt.

So we can compare:

```text
full replay cache: [new system freshly computed] [old units freshly recomputed under new system]
surgery cache:     [new system freshly computed] [old unit cache preserved/reindexed]
```

Possible divergence metrics:

- **Cache tensor difference:** compare preserved/replayed K/V tensors with L2 or cosine distance.
- **Next-token logits difference:** run one next-token step and compare logits.
- **Distribution difference:** compare next-token probabilities with KL divergence or top-k overlap.
- **Behavior difference:** generate from both states and compare text/audio outputs.

For model debugging, logits/probability divergence is usually more informative than text-only comparison because text sampling can hide or amplify small differences.

In [ ]:
def l2_relative(a, b, eps=1e-8):
    return (a - b).norm() / (b.norm() + eps)

def cosine_distance(a, b, eps=1e-8):
    a_flat = a.reshape(-1)
    b_flat = b.reshape(-1)
    cosine = torch.dot(a_flat, b_flat) / (a_flat.norm() * b_flat.norm() + eps)
    return 1 - cosine

def kl_divergence_from_logits(logits_a, logits_b, eps=1e-8):
    # KL(P_a || P_b)
    p = torch.softmax(logits_a, dim=-1)
    log_p = torch.log(p + eps)
    log_q = torch.log_softmax(logits_b, dim=-1)
    return torch.sum(p * (log_p - log_q))

print('Metric helpers ready: l2_relative, cosine_distance, kl_divergence_from_logits')

### Toy Divergence Demonstration

The fake `StreamDecoder` test above intentionally preserves old unit **values** exactly. That is good for verifying cache surgery mechanics, but it does not show full-replay divergence very clearly.

The next toy example creates context-sensitive unit vectors:

- Old units depend on the old system vector.
- Full replay units depend on the new system vector.
- Surgery keeps the old units after changing the system vector.

This makes the approximation visible.

In [ ]:
torch.manual_seed(7)

old_system = torch.randn(64)
new_system = old_system + 0.25 * torch.randn(64)
raw_units = torch.randn(512, 64)

# Toy context influence: in a real transformer, each unit representation is affected by attention to prior context.
old_unit_cache = raw_units + 0.20 * old_system
full_replay_unit_cache = raw_units + 0.20 * new_system
surgery_unit_cache = old_unit_cache.clone()

print('Unit cache divergence: surgery vs full replay')
print('relative L2:      ', float(l2_relative(surgery_unit_cache, full_replay_unit_cache)))
print('cosine distance:  ', float(cosine_distance(surgery_unit_cache, full_replay_unit_cache)))

# Toy next-token logits from the cache. Real models would use the cache inside the LLM forward pass.
vocab_projection = torch.randn(64, 1000)
full_replay_logits = full_replay_unit_cache[-1] @ vocab_projection
surgery_logits = surgery_unit_cache[-1] @ vocab_projection

kl = kl_divergence_from_logits(surgery_logits, full_replay_logits)
print('\nNext-token distribution divergence')
print('KL(surgery || full_replay):', float(kl))

full_top = torch.topk(full_replay_logits, 10).indices.tolist()
surgery_top = torch.topk(surgery_logits, 10).indices.tolist()
overlap = len(set(full_top) & set(surgery_top))
print('top-10 overlap:', overlap, '/ 10')
print('full replay top-10:', full_top)
print('surgery top-10:   ', surgery_top)

### How To Compare Divergence On The Real Model

For the real MiniCPM model, the best comparison is:

1. Start from the same conversation history.
2. Path A: change prompt with cache surgery.
3. Path B: clear cache and replay the same raw history under the new prompt.
4. Feed the exact same next input to both paths.
5. Compare next-token logits/probabilities and generated outputs.

The hard part is step 3: exact replay requires saving the raw replayable inputs for prior units, including audio/video frames and text units. If you only saved K/V cache, you cannot reconstruct a mathematically exact replay later.

A practical evaluation table could track:

```text
cache_length
update_latency_ms
peak_gpu_memory_mb
next_token_kl
next_token_top1_same
next_token_top10_overlap
generated_text_similarity
human pass/fail for behavior
```

The code below is a skeleton. It is not run automatically because it depends on the real model and on having a replayable conversation fixture.

In [ ]:
RUN_REAL_DIVERGENCE_SKETCH = False

if RUN_REAL_DIVERGENCE_SKETCH:
    # Pseudocode sketch. You need real model setup and replayable session inputs.
    #
    # processor_a = UnifiedProcessor.from_pretrained(model_path=MODEL_PATH, pt_path=PT_PATH)
    # processor_b = UnifiedProcessor.from_pretrained(model_path=MODEL_PATH, pt_path=PT_PATH)
    #
    # duplex_a = processor_a.set_duplex_mode(ref_audio_path=REF_AUDIO)
    # duplex_b = processor_b.set_duplex_mode(ref_audio_path=REF_AUDIO)
    #
    # duplex_a.prepare(system_prompt_text=OLD_PROMPT)
    # replay_history_into(duplex_a, HISTORY)
    # duplex_a.update_system_prompt(system_prompt_text=NEW_PROMPT)  # cache surgery path
    #
    # duplex_b.prepare(system_prompt_text=NEW_PROMPT)
    # replay_history_into(duplex_b, HISTORY)  # full replay path
    #
    # logits_a = run_one_next_token_logits(duplex_a, NEXT_INPUT)
    # logits_b = run_one_next_token_logits(duplex_b, NEXT_INPUT)
    #
    # print('KL:', kl_divergence_from_logits(logits_a, logits_b))
    # print('top-10 overlap:', topk_overlap(logits_a, logits_b, k=10))
    print('Fill in real model setup, replay_history_into, and run_one_next_token_logits for your fixture.')
else:
    print('Set RUN_REAL_DIVERGENCE_SKETCH = True only after preparing a real replay fixture.')

## 10. What This Proves And What It Does Not Prove

This proves the current implementation can surgically rebuild the protected prompt span while preserving later unit cache values. It also verifies the bookkeeping fields that tell the decoder where prefix, previous summary, suffix, and unit cache sections live.

It does not prove mathematical equivalence to replaying the entire conversation from scratch under the new system prompt.

Why not?

A cached unit was originally computed while attending to the old system prompt. When we preserve that unit cache, we keep its old internal K/V representation. Future tokens then attend to:

```text
[new system cache] + [preserved old unit cache]
```

This is useful for speed and continuity, but it is cache surgery rather than full replay.

Exact recaching would require storing replayable raw inputs for every prior unit and running them again after the new prompt. That would be slower and use more memory/storage, but it is the route if exact replay semantics become necessary.

## 11. Optional: Run The Lightweight Repo Test In Colab

After the notebook cells above make sense, run the real test file. It uses the same fake-model idea, but as an automated regression check.

In [ ]:
if REPO_PATH.exists():
    %cd {REPO_PATH}
    !PYTHONPATH=src python -m pytest -q tests/test_duplex_system_prompt_update.py -s
else:
    print('Clone/upload the repo first, then run this cell.')

## 12. Optional: Real Model Smoke Shape

Only run this when your Colab runtime has the real model dependencies, model path, and enough GPU memory. This is intentionally a skeleton so you can fill in your actual model paths.

In [ ]:
# Optional real-model sketch. Leave disabled until your paths and deps are ready.
RUN_REAL_MODEL = False

if RUN_REAL_MODEL:
    from minicpmo_demo.core.processors.unified import UnifiedProcessor

    MODEL_PATH = '/content/path/to/MiniCPM-o-model'
    PT_PATH = '/content/path/to/model.pt'
    REF_AUDIO = '/content/path/to/ref.wav'

    processor = UnifiedProcessor.from_pretrained(model_path=MODEL_PATH, pt_path=PT_PATH)
    duplex = processor.set_duplex_mode(ref_audio_path=REF_AUDIO)
    duplex.prepare(system_prompt_text='You are concise.')

    before = processor.kv_cache_length
    ok = duplex.update_system_prompt(system_prompt_text='You are concise and mention tool changes when relevant.')
    after = processor.kv_cache_length

    print({'updated': ok, 'cache_before': before, 'cache_after': after})
else:
    print('Set RUN_REAL_MODEL = True only after configuring real model paths.')